# Askvocate AI Service — Final Hybrid Router
## Local vector model is PRIMARY; Gemini is only a rescue for low-confidence cases

### Behavior
- Step 1: Local classifier runs first for every prompt (any language)
- Step 2: Language detection adjusts the confidence threshold
- Step 3: If local is confident → return local (no Gemini call)
- Step 4: If local unsure AND Gemini available → try Gemini
- Step 5: If Gemini unavailable → return local with needs_clarification
- Step 6: Lawyer matching is always local (taxonomy + embedding rerank)


In [1]:
# ---------- Setup ----------
import os, json, re, time, hashlib, pickle
from functools import lru_cache
from pathlib import Path

import pandas as pd
import numpy as np

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"))
except Exception:
    pass

from sentence_transformers import SentenceTransformer, util

print("Loading datasets from ../datasets/ ...")
cases   = pd.read_csv('../datasets/cases.csv')
lawyers = pd.read_csv('../datasets/advocate_details.csv')
queries = pd.read_csv('../datasets/test_queries.csv')

print(f"Cases shape   : {cases.shape}")
print(f"Lawyers shape : {lawyers.shape}")
print(f"Queries shape : {queries.shape}")

# Verify env loaded
_key = os.getenv("GEMINI_API_KEY")
print(f"GEMINI_API_KEY loaded: {bool(_key)}" + (f" (len={len(_key)})" if _key else ""))

c:\Users\shiva\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading datasets from ../datasets/ ...
Cases shape   : (10000, 31)
Lawyers shape : (7000, 44)
Queries shape : (666, 6)
GEMINI_API_KEY loaded: True (len=53)


In [2]:
# ---------- Domain taxonomy ----------
DOMAIN_TAXONOMY = {
    "Administrative Law": {"primary": ["Administrative Law"], "secondary": ["Constitutional Law", "Civil Litigation"]},
    "Arbitration": {"primary": ["Arbitration and Mediation"], "secondary": ["Commercial Law", "Corporate Law"]},
    "Banking & Finance": {"primary": ["Banking and Finance"], "secondary": ["Corporate Law", "Securities Law"]},
    "Child Protection": {"primary": ["Child Protection Law"], "secondary": ["Family Law", "Constitutional Law"]},
    "Constitutional Law": {"primary": ["Constitutional Law"], "secondary": ["Human Rights Law", "Public Interest Litigation"]},
    "Consumer Protection": {"primary": ["Consumer Law"], "secondary": ["Civil Litigation", "Commercial Law"]},
    "Contract & Agreement": {"primary": ["Contract Law"], "secondary": ["Commercial Law", "Civil Litigation"]},
    "Corporate & Commercial": {"primary": ["Corporate Law"], "secondary": ["Banking and Finance", "Securities Law"]},
    "Criminal Law": {"primary": ["Criminal Law"], "secondary": ["Constitutional Law"]},
    "Cybercrime & IT": {"primary": ["Cyber Law"], "secondary": ["Data Protection and Privacy", "Information Technology Law"]},
    "Data Privacy": {"primary": ["Data Protection and Privacy"], "secondary": ["Information Technology Law", "Cyber Law"]},
    "Divorce & Matrimonial": {"primary": ["Matrimonial Law"], "secondary": ["Family Law"]},
    "Education Law": {"primary": ["Education Law"], "secondary": ["Constitutional Law", "Administrative Law"]},
    "Employment & Labour": {"primary": ["Labour and Employment Law"], "secondary": ["Civil Litigation"]},
    "Environmental Law": {"primary": ["Environmental Law"], "secondary": ["Public Interest Litigation", "Constitutional Law"]},
    "Family & Succession": {"primary": ["Family Law"], "secondary": ["Matrimonial Law"]},
    "Human Rights & PIL": {"primary": ["Human Rights Law"], "secondary": ["Public Interest Litigation", "Constitutional Law"]},
    "Immigration & Citizenship": {"primary": ["Immigration Law"], "secondary": ["Constitutional Law", "Civil Litigation"]},
    "Insolvency & Bankruptcy": {"primary": ["Insolvency and Bankruptcy"], "secondary": ["Banking and Finance", "Corporate Law"]},
    "Intellectual Property": {"primary": ["Intellectual Property"], "secondary": ["Patent Law", "Trademark Law", "Copyright Law"]},
    "Media & Defamation": {"primary": ["Media and Entertainment Law"], "secondary": ["Civil Litigation", "Criminal Law"]},
    "Medical Negligence": {"primary": ["Medical Negligence Law"], "secondary": ["Consumer Law", "Civil Litigation"]},
    "Motor Accident Claims": {"primary": ["Motor Accident Claims Law"], "secondary": ["Personal Injury Law", "Civil Litigation"]},
    "Property & Land": {"primary": ["Property Law"], "secondary": ["Real Estate Law", "Civil Litigation"]},
    "Real Estate & Housing": {"primary": ["Real Estate Law"], "secondary": ["Property Law", "Civil Litigation"]},
    "Tax & GST": {"primary": ["Tax Law"], "secondary": ["GST Law", "Corporate Law"]},
    "White Collar Crime": {"primary": ["White Collar Crime"], "secondary": ["Criminal Law", "Banking and Finance"]},
}

EXPANDED_DESCRIPTIONS = {
    "Administrative Law": "Administrative Law, government officer action, public authority order, writ petition, sarkari notice, sarkari adhikari, IAS IPS complaint, official service dispute, administrative tribunal.",
    "Arbitration": "Arbitration and Mediation, commercial contract dispute resolution, out of court settlement, arbitrator award, compromise agreement, conciliation, panchayat.",
    "Banking & Finance": "Banking & Finance, bank loan default, cheque bounce Section 138 NI Act, NPA recovery, bank account frozen, loan recovery notice, credit card fraud, SARFAESI, karza, karz nahi chuka.",
    "Child Protection": "Child Protection, child custody, adoption, minor guardianship, POCSO act, child abuse, child welfare committee, Bachpan, bacha custody, bachche ka adhikar.",
    "Constitutional Law": "Constitutional Law, fundamental rights violation, writ petition Article 32 Article 226, high court supreme court writ, government law challenge, habeas corpus, mandamus, maulik adhikar.",
    "Consumer Protection": "Consumer Protection, consumer court complaint, defective product, service deficiency, refund claim, consumer forum, online shopping refund, warranty claim, grahak suraksha.",
    "Contract & Agreement": "Contract & Agreement, breach of contract, agreement violation, business agreement non-compliance, NDA breach, service agreement dispute, contract enforcement, samjhauta.",
    "Corporate & Commercial": "Corporate & Commercial, company incorporation, shareholder dispute, merger acquisition, corporate governance, MCA ROC filing, director dispute, partnership agreement, kharidaar.",
    "Criminal Law": "Criminal Law, FIR police station, bail application, murder theft robbery assault, criminal trial, police harassment, crime, jail, arrest, anticipatory bail, IPC sections, giraftari, hakim, mukadma.",
    "Cybercrime & IT": "Cybercrime & IT, online fraud, cyber fraud, account hacked, phishing scam, unauthorized bank transaction, OTP fraud, cyber bullying, IT act 66D, cyber cell complaint, paise kat gaye online, UPI fraud, ऑनलाइन फ्रॉड, ऑनलाइन ठगी, बैंक खाता हैक, साइबर अपराध, इंटरनेट ठगी.",
    "Data Privacy": "Data Privacy, data leak, personal data breach, privacy violation, unauthorized data sharing, IT act privacy breach, customer data stolen, vyaktigat jankari.",
    "Divorce & Matrimonial": "Divorce & Matrimonial, divorce petition, matrimonial dispute, alimony maintenance claim, Section 498A dowry harassment, spouse separation, talaq, pati patni vivad, stridhan, तलाक, गुजारा भत्ता, दहेज, पति पत्नी.",
    "Education Law": "Education Law, school college admission dispute, university degree recognition, exam fee dispute, student rights, ragging complaint, seat allocation dispute, shiksha, padhai.",
    "Employment & Labour": "Employment & Labour, wrongful termination, salary non-payment, employee gratuity, fired without notice, labor court, PF claim, resignation dispute, job termination, naukri, kaam se nikaal diya, salary nahi mili, नौकरी, वेतन, सैलरी नहीं मिली, नौकरी से निकाल दिया, ग्रेच्युटी.",
    "Environmental Law": "Environmental Law, pollution complaint, NGT National Green Tribunal order, environmental violation, industrial waste dumping, air water pollution, forest land violation, pradushan.",
    "Family & Succession": "Family & Succession, property inheritance dispute, ancestral property partition, will probate, family property division, succession certificate, virasat, baap dada ki zameen, virasat ka jhagda, जमीन विरासत.",
    "Human Rights & PIL": "Human Rights & PIL, human rights violation, public interest litigation, police brutality, custodial violence, civil liberties, human rights commission, manav adhikar.",
    "Immigration & Citizenship": "Immigration & Citizenship, visa rejection, passport dispute, citizenship application, foreign visa appeal, deportation notice, NRI legal issue, visa samasya.",
    "Insolvency & Bankruptcy": "Insolvency & Bankruptcy, NCLT insolvency resolution, IBC 2016, bankruptcy proceedings, corporate debt default, insolvency professional, liquidation, diwaliya.",
    "Intellectual Property": "Intellectual Property, patent infringement, trademark violation, copyright theft, brand IP dispute, logo copyright, patent registration breach, brand nakal.",
    "Media & Defamation": "Media & Defamation, defamation notice Section 499 500, media libel, slander, reputation damage, newspaper news media dispute, manhani notice, badnami.",
    "Medical Negligence": "Medical Negligence, doctor negligence, hospital malpractice, wrong surgery, medical death compensation, patient wrong treatment, medical council complaint, doctor ki laaparwahi.",
    "Motor Accident Claims": "Motor Accident Claims, road accident compensation, MACT claim tribunal, hit and run, motor vehicle accident, bike car accident claim, insurance claim compensation, gadi accident, gaadi takkar, leg fracture accident, सड़क दुर्घटना, गाड़ी की टक्कर, मुआवजा, एक्सीडेंट.",
    "Property & Land": "Property & Land, property ownership, land dispute, zameen kabza, property registry, boundary dispute, illegal land encroachment, plot registry dispute, khet zameen, जमीन का विवाद, कब्जा, रजिस्ट्री.",
    "Real Estate & Housing": "Real Estate & Housing, landlord tenant dispute, security deposit refund, makaan malik, kirayedar, builder flat possession delay, RERA complaint, PG deposit refund, eviction notice, landlord paisa nahi de raha, deposit wapas nahi, मकान मालिक, किरायेदार, जमा राशि, फ्लैट विवाद.",
    "Tax & GST": "Tax & GST, income tax notice, GST audit, tax evasion, income tax refund claim, IT department dispute, GST tribunal, tax assessment, aaykar, kar chori.",
    "White Collar Crime": "White Collar Crime, money laundering PMLA, ED investigation, financial scam fraud, forgery embezzlement, CBI case, corruption prevention of corruption act, ghotala, bharstachar.",
}

# --- English/Hinglish boost keywords ---
BOOSTS = {
    "Real Estate & Housing":     ["security deposit", "landlord", "kirayedar", "makaan malik", "rent deposit", "eviction", "tenant", "builder possession", "rera"],
    "Employment & Labour":       ["gratuity", "fired without notice", "wrongful termination", "salary non-payment", "pf claim", "labour court", "kaam se nikaal", "salary nahi mili", "naukri se nikaal"],
    "Cybercrime & IT":           ["online fraud", "otp fraud", "upi fraud", "account hacked", "phishing", "cyber fraud", "unauthorized transaction", "paise kat gaye"],
    "Motor Accident Claims":     ["road accident", "mact", "hit and run", "bike accident", "car accident", "leg fracture", "gaadi takkar", "gaadi accident"],
    "Divorce & Matrimonial":     ["divorce", "alimony", "498a", "dowry", "talaq", "matrimonial", "pati patni", "stridhan"],
    "Banking & Finance":         ["cheque bounce", "138 ni act", "loan default", "sarfaesi", "credit card fraud"],
    "Consumer Protection":       ["defective product", "consumer court", "refund claim", "grahak suraksha"],
    "Criminal Law":              ["fir", "bail", "anticipatory bail", "murder", "robbery", "assault", "giraftari", "mukadma"],
    "White Collar Crime":        ["money laundering", "pmla", "ed investigation", "cbi case", "bharstachar"],
    "Medical Negligence":        ["medical negligence", "doctor negligence", "wrong surgery", "hospital malpractice", "doctor ki laaparwahi"],
    "Tax & GST":                 ["income tax notice", "gst audit", "tax evasion", "it department"],
    "Property & Land":           ["zameen kabza", "land encroachment", "registry dispute"],
}

# --- Devanagari boost keywords (added per domain) ---
BOOSTS_DEVANAGARI = {
    "Real Estate & Housing":   ["मकान मालिक", "किरायेदार", "जमा राशि", "किराया", "फ्लैट", "इविक्शन"],
    "Employment & Labour":     ["नौकरी", "वेतन", "सैलरी", "ग्रेच्युटी", "बर्खास्त", "निकाल दिया", "तनख्वाह"],
    "Cybercrime & IT":         ["ऑनलाइन फ्रॉड", "ऑनलाइन ठगी", "बैंक खाता", "साइबर", "इंटरनेट ठगी", "ओटीपी"],
    "Motor Accident Claims":   ["सड़क दुर्घटना", "दुर्घटना", "गाड़ी की टक्कर", "मुआवजा", "एक्सीडेंट"],
    "Divorce & Matrimonial":   ["तलाक", "गुजारा भत्ता", "दहेज", "पति पत्नी", "शादी"],
    "Criminal Law":            ["एफआईआर", "जमानत", "गिरफ्तारी", "मुकदमा", "पुलिस"],
    "Property & Land":         ["जमीन", "कब्जा", "रजिस्ट्री", "संपत्ति"],
    "Medical Negligence":      ["डॉक्टर", "अस्पताल", "इलाज", "लापरवाही"],
    "Tax & GST":               ["आयकर", "कर", "जीएसटी", "टैक्स"],
    "Family & Succession":     ["विरासत", "उत्तराधिकार", "पैतृक संपत्ति"],
    "Banking & Finance":       ["बैंक", "चेक", "लोन", "कर्ज"],
    "Consumer Protection":     ["उपभोक्ता", "खराब सामान", "रिफंड"],
    "White Collar Crime":      ["भ्रष्टाचार", "घोटाला", "मनी लॉन्ड्रिंग"],
    "Data Privacy":            ["डेटा", "निजता", "प्राइवेसी"],
    "Child Protection":        ["बच्चा", "बच्चे", "अभिरक्षा", "गोद"],
    "Human Rights & PIL":      ["मानवाधिकार", "पुलिस अत्याचार"],
}

# --- Devanagari → Latin transliteration for common legal terms ---
TRANSLIT = {
    "ऑनलाइन": "online", "फ्रॉड": "fraud", "ठगी": "fraud", "धोखाधड़ी": "fraud",
    "बैंक": "bank", "खाता": "account", "ओटीपी": "otp", "साइबर": "cyber",
    "मकान": "makaan", "किराया": "kiraya", "किरायेदार": "kirayedar",
    "मालिक": "malik", "फ्लैट": "flat", "जमा": "deposit", "राशि": "amount",
    "नौकरी": "naukri", "वेतन": "salary", "सैलरी": "salary", "तनख्वाह": "salary",
    "ग्रेच्युटी": "gratuity", "बर्खास्त": "fired", "निकाल": "fired",
    "दुर्घटना": "accident", "एक्सीडेंट": "accident", "गाड़ी": "gaadi",
    "टक्कर": "takkar", "मुआवजा": "compensation",
    "तलाक": "divorce", "दहेज": "dowry", "गुजारा": "alimony", "भत्ता": "alimony",
    "शादी": "marriage", "पति": "husband", "पत्नी": "wife",
    "पुलिस": "police", "एफआईआर": "fir", "जमानत": "bail", "गिरफ्तारी": "arrest",
    "जमीन": "zameen", "कब्जा": "kabza", "रजिस्ट्री": "registry", "संपत्ति": "property",
    "डॉक्टर": "doctor", "अस्पताल": "hospital", "इलाज": "treatment", "लापरवाही": "negligence",
    "आयकर": "income tax", "कर": "tax", "जीएसटी": "gst", "टैक्स": "tax",
    "विरासत": "inheritance", "उत्तराधिकार": "succession", "पैतृक": "ancestral",
    "चेक": "cheque", "लोन": "loan", "कर्ज": "loan",
    "उपभोक्ता": "consumer", "रिफंड": "refund",
    "भ्रष्टाचार": "corruption", "घोटाला": "scam",
    "डेटा": "data", "निजता": "privacy", "प्राइवेसी": "privacy",
    "बच्चा": "child", "बच्चे": "child", "अभिरक्षा": "custody", "गोद": "adoption",
    "मानवाधिकार": "human rights", "अत्याचार": "brutality",
}

# ---------- Special routing rules (override embedding when keyword is unambiguous) ----------
# These take priority over the embedding classifier — the phrase is too
# sensitive/important to leave to similarity scoring.
SPECIAL_ROUTES = {
    # Domestic violence → Criminal Law (PWDVA is enforced via criminal courts)
"domestic violence":     "Criminal Law",
"घरेलू हिंसा":             "Criminal Law",
"gharelu hinsa":         "Criminal Law",
"husband beats me":      "Criminal Law",
"pati maarta hai":       "Criminal Law",
"pati ne maara":         "Criminal Law",
"मुझे मारा":               "Criminal Law",
"मारपीट":                 "Criminal Law",
"pati ne mujhe maara":   "Criminal Law",
"pati maar raha hai":    "Criminal Law",
"husband maar raha":     "Criminal Law",

# Forced eviction from matrimonial home → Criminal Law (S.19 PWDVA is enforced criminally)
"ghar se nikaal diya":   "Criminal Law",
"घर से निकाल दिया":         "Criminal Law",

# Pure stridhan recovery stays civil
"stridhan":              "Family & Succession",
"स्त्रीधन":                "Family & Succession",

# Dowry/498A → Criminal Law (already correct)
"498a":                  "Criminal Law",
"dowry":                 "Criminal Law",
"दहेज":                   "Criminal Law",
}

print(f"Special routes loaded: {len(SPECIAL_ROUTES)} triggers")


def normalize_prompt(prompt: str) -> str:
    """Transliterate Devanagari legal terms to Latin for boosting."""
    p = prompt.lower()
    for dev, lat in TRANSLIT.items():
        p = p.replace(dev, lat)
    return p


def detect_language(prompt: str) -> str:
    """Return 'devanagari', 'hinglish', or 'english'."""
    if any('\u0900' <= c <= '\u097F' for c in prompt):
        return "devanagari"
    hinglish_markers = {"mera", "meri", "nahi", "hai", "raha", "rahi", "kya",
                        "kyun", "wapas", "de", "di", "kar", "se", "ka", "ki",
                        "ko", "aur", "par", "tha", "thi", "hua", "gaya"}
    words = set(re.findall(r'\b\w+\b', prompt.lower()))
    if len(words & hinglish_markers) >= 2:
        return "hinglish"
    return "english"


# Build enriched domain descriptions
DOMAIN_DESCRIPTIONS = {}
for d in DOMAIN_TAXONOMY.keys():
    kw = ' '.join(cases[cases['legal_domain'] == d]['search_keywords'].dropna().tolist()[:30])
    DOMAIN_DESCRIPTIONS[d] = f"{EXPANDED_DESCRIPTIONS.get(d, d)} Keywords from cases: {kw}"

sparse = [d for d in DOMAIN_TAXONOMY if len(DOMAIN_DESCRIPTIONS[d].split()) < 30]
if sparse:
    print(f"⚠️  Sparse descriptions: {sparse}")
else:
    print("Enriched domain descriptions loaded ✓")
print(f"Transliteration map: {len(TRANSLIT)} terms")
print(f"Devanagari boosts : {sum(len(v) for v in BOOSTS_DEVANAGARI.values())} keywords across {len(BOOSTS_DEVANAGARI)} domains")

Special routes loaded: 18 triggers
Enriched domain descriptions loaded ✓
Transliteration map: 69 terms
Devanagari boosts : 68 keywords across 16 domains


In [3]:
# ---------- Local vector model (PRIMARY path) ----------
EMBEDDING_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
CACHE_FILE = '../datasets/domain_embeddings_v3.pkl'

print(f"Initializing multilingual local embedder: {EMBEDDING_MODEL_NAME} ...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

domain_keys = sorted(list(DOMAIN_TAXONOMY.keys()))

# Hash includes descriptions + boosts → cache auto-invalidates on any edit
taxonomy_payload = json.dumps({
    "domains": domain_keys,
    "descriptions": {k: DOMAIN_DESCRIPTIONS[k] for k in domain_keys},
    "boosts": BOOSTS,
    "boosts_dev": BOOSTS_DEVANAGARI,
}, sort_keys=True)
taxonomy_hash = hashlib.md5(taxonomy_payload.encode()).hexdigest()[:8]

cached = None
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f:
        cached = pickle.load(f)
    if cached.get("model") == EMBEDDING_MODEL_NAME and cached.get("tax_hash") == taxonomy_hash:
        domain_embeddings = cached["embeddings"]
        print(f"Loaded cached domain embeddings ({domain_embeddings.shape}) ✓")
    else:
        print(f"Cache mismatch ({cached.get('tax_hash')} → {taxonomy_hash}). Rebuilding ...")
        cached = None

if cached is None:
    print("Building domain embeddings ...")
    domain_texts = [f"{k}: {DOMAIN_DESCRIPTIONS[k]}" for k in domain_keys]
    domain_embeddings = embedder.encode(domain_texts, convert_to_numpy=True, show_progress_bar=True)
    with open(CACHE_FILE, "wb") as f:
        pickle.dump({
            "model": EMBEDDING_MODEL_NAME,
            "tax_hash": taxonomy_hash,
            "embeddings": domain_embeddings
        }, f)
    print(f"Saved embeddings to {CACHE_FILE} ✓")

print(f"Local embedding matrix shape: {domain_embeddings.shape}")


@lru_cache(maxsize=2048)
def _cached_query_embedding(prompt: str):
    return embedder.encode(prompt, convert_to_numpy=True)


def _apply_keyword_boost(sims: np.ndarray, prompt: str) -> np.ndarray:
    """Boost domains for both Latin and Devanagari keyword hits."""
    p_lat = normalize_prompt(prompt)          # Devanagari → Latin
    p_raw = prompt.lower()                    # for Devanagari keywords
    sims = sims.copy()

    for domain, keywords in BOOSTS.items():
        if domain not in domain_keys:
            continue
        if any(kw in p_lat for kw in keywords):
            sims[domain_keys.index(domain)] += 0.05

    for domain, keywords in BOOSTS_DEVANAGARI.items():
        if domain not in domain_keys:
            continue
        if any(kw in p_raw for kw in keywords):
            sims[domain_keys.index(domain)] += 0.05

    return sims


def _calibrate_confidence(sims: np.ndarray, temperature: float = 0.05) -> np.ndarray:
    z = (sims - sims.max()) / temperature
    e = np.exp(z)
    return e / e.sum()


def vector_extract_intent(user_prompt: str) -> dict:
    """
    Local classifier with:
      - SPECIAL_ROUTES override (highest priority)
      - top-3 domains + margin
      - calibrated confidence
      - language tag
    """
    t0 = time.time()
    prompt_lower = user_prompt.lower()

    # --- Step 0: special routes (keyword overrides) ---
    for trigger, forced_domain in SPECIAL_ROUTES.items():
        if trigger in prompt_lower:
            latency_ms = (time.time() - t0) * 1000
            return {
                "primary_domain": forced_domain,
                "confidence": 0.99,
                "raw_cosine": 1.0,
                "margin": 1.0,
                "top3_domains": [(forced_domain, 1.0)],
                "ambiguous": False,
                "language": detect_language(user_prompt),
                "lawyer_practice_areas": DOMAIN_TAXONOMY[forced_domain]["primary"],
                "latency_ms": round(latency_ms, 2),
                "source": "special_route",
                "matched_trigger": trigger,
            }

    # --- Step 1: normal embedding + boost path ---
    query_vec = _cached_query_embedding(user_prompt)
    raw_sims = util.cos_sim(query_vec, domain_embeddings).numpy()[0]
    sims = _apply_keyword_boost(raw_sims, user_prompt)

    top3_idx = np.argsort(sims)[::-1][:3]
    top3 = [(domain_keys[i], float(sims[i])) for i in top3_idx]
    margin = float(sims[top3_idx[0]] - sims[top3_idx[1]])
    probs = _calibrate_confidence(sims)
    calibrated_conf = float(probs[top3_idx[0]])
    best_domain = domain_keys[top3_idx[0]]
    latency_ms = (time.time() - t0) * 1000

    return {
        "primary_domain": best_domain,
        "confidence": round(calibrated_conf, 4),
        "raw_cosine": round(float(sims[top3_idx[0]]), 4),
        "margin": round(margin, 4),
        "top3_domains": top3,
        "ambiguous": margin < 0.05,
        "language": detect_language(user_prompt),
        "lawyer_practice_areas": DOMAIN_TAXONOMY[best_domain]["primary"],
        "latency_ms": round(latency_ms, 2),
        "source": "local_vector_embedding"
    }


print("Local offline vector classifier initialized ✓")

Initializing multilingual local embedder: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...
Loaded cached domain embeddings ((27, 384)) ✓
Local embedding matrix shape: (27, 384)
Local offline vector classifier initialized ✓


In [4]:
# ---------- Gemini availability (circuit breaker only — no live probe) ----------
import os, time, json
from pathlib import Path

try:
    from dotenv import load_dotenv
    for candidate in [Path("../.env"), Path(".env")]:
        if candidate.exists():
            load_dotenv(candidate, override=True)
            break
except Exception as e:
    print(f"dotenv load failed: {e}")

GEMINI_MODEL_CANDIDATES = [
    "gemini-3.5-flash",
    "gemini-3.6-flash",
    "gemini-3.8-flash",
]

GEMINI_CLIENT = None
_CIRCUIT = {"failures": 0, "open_until": 0, "last_failure": 0}
CIRCUIT_THRESHOLD = 3
CIRCUIT_COOLDOWN = 120

try:
    from google import genai
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        print("⚠️  GEMINI_API_KEY not found. Local-only mode.")
    elif len(GEMINI_API_KEY) <= 15 or GEMINI_API_KEY.startswith("your_"):
        print(f"⚠️  GEMINI_API_KEY looks invalid. Local-only mode.")
    else:
        GEMINI_CLIENT = genai.Client(api_key=GEMINI_API_KEY)
        print(f"Gemini client initialized ✓ (key len={len(GEMINI_API_KEY)})")
except Exception as e:
    print(f"Gemini SDK unavailable: {e}")


def check_gemini_available() -> bool:
    if GEMINI_CLIENT is None:
        return False
    if time.time() < _CIRCUIT["open_until"]:
        return False
    return True


def _trip_circuit(reason: str = ""):
    _CIRCUIT["failures"] += 1
    _CIRCUIT["last_failure"] = time.time()
    if _CIRCUIT["failures"] >= CIRCUIT_THRESHOLD:
        _CIRCUIT["open_until"] = time.time() + CIRCUIT_COOLDOWN
        print(f"⚠️  Gemini circuit OPEN for {CIRCUIT_COOLDOWN}s ({reason})")


def _reset_circuit():
    _CIRCUIT["failures"] = 0
    _CIRCUIT["open_until"] = 0


print("Gemini availability check ready ✓")

Gemini client initialized ✓ (key len=53)
Gemini availability check ready ✓


In [5]:
# ---------- Gemini domain classification ----------
VALID_DOMAINS = set(DOMAIN_TAXONOMY.keys())


def extract_intent_with_gemini(user_prompt: str) -> dict:
    if GEMINI_CLIENT is None:
        return {"status": "unavailable", "source": "gemini_unavailable"}

    any_hard_failure = False
    failures_this_call = 0

    for model_name in GEMINI_MODEL_CANDIDATES:
        try:
            prompt = f"""
You are a legal intent router for Askvocate.
Pick exactly one domain from this list:
{', '.join(VALID_DOMAINS)}

Return ONLY valid JSON in this exact format:
{{
  "primary_domain": "one item from the list",
  "confidence": 0.0,
  "reason": "brief explanation",
  "language": "English/Hindi/Hinglish"
}}

User query: {user_prompt}
"""
            response = GEMINI_CLIENT.models.generate_content(model=model_name, contents=prompt)
            content = getattr(response, "text", None) or str(response)
            parsed = json.loads(content)

            if parsed.get("primary_domain") not in VALID_DOMAINS:
                print(f"⚠️  Gemini invalid domain: {parsed.get('primary_domain')}")
                failures_this_call += 1
                continue

            parsed["source"] = f"gemini:{model_name}"
            parsed["status"] = "ok"
            _reset_circuit()
            return parsed

        except json.JSONDecodeError as e:
            print(f"Gemini JSON parse failed ({model_name}): {e}")
            failures_this_call += 1
            continue

        except Exception as e:
            failures_this_call += 1
            msg = str(e).lower()
            if any(x in msg for x in ("429", "quota", "exhausted", "rate limit")):
                print(f"Gemini quota exhausted ({model_name})")
                any_hard_failure = True
            elif any(x in msg for x in ("503", "unavailable")):
                print(f"Gemini unavailable ({model_name})")
                any_hard_failure = True
            elif "not found" in msg or "404" in msg:
                print(f"Gemini model missing ({model_name})")
                any_hard_failure = True
            else:
                print(f"Gemini request failed ({model_name}): {e}")
            continue

    # All models exhausted
    if any_hard_failure or failures_this_call >= len(GEMINI_MODEL_CANDIDATES):
        _trip_circuit("all models failed")

    return {"status": "fallback", "source": "gemini_unavailable"}


print("Gemini classifier ready ✓")

Gemini classifier ready ✓


In [6]:
# ---------- Groq fallback (used when Gemini fails) ----------
import os

GROQ_CLIENT = None
GROQ_MODEL = "llama-3.3-70b-versatile"
GROQ_CIRCUIT = {"failures": 0, "open_until": 0}
GROQ_COOLDOWN = 120
GROQ_THRESHOLD = 3

try:
    from groq import Groq
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    if GROQ_API_KEY and len(GROQ_API_KEY) > 15:
        GROQ_CLIENT = Groq(api_key=GROQ_API_KEY)
        print(f"Groq client initialized ✓ (key len={len(GROQ_API_KEY)})")
    else:
        print("⚠️  GROQ_API_KEY missing or invalid. Groq fallback disabled.")
except ImportError:
    print("⚠️  groq package not installed. Run: pip install groq")
except Exception as e:
    print(f"Groq init failed: {e}")


def check_groq_available() -> bool:
    if GROQ_CLIENT is None:
        return False
    if time.time() < GROQ_CIRCUIT["open_until"]:
        return False
    return True


def extract_intent_with_groq(user_prompt: str) -> dict:
    if GROQ_CLIENT is None:
        return {"status": "unavailable", "source": "groq_unavailable"}

    try:
        prompt = f"""You are a legal intent router for Askvocate.
Pick exactly one domain from this list:
{', '.join(VALID_DOMAINS)}

Return ONLY valid JSON (no markdown fences) with this exact shape:
{{
  "primary_domain": "one item from the list",
  "confidence": 0.0,
  "reason": "brief explanation",
  "language": "English/Hindi/Hinglish"
}}

User query: {user_prompt}"""

        resp = GROQ_CLIENT.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        content = resp.choices[0].message.content
        parsed = json.loads(content)

        if parsed.get("primary_domain") not in VALID_DOMAINS:
            print(f"⚠️  Groq invalid domain: {parsed.get('primary_domain')}")
            return {"status": "fallback", "source": "groq_invalid"}

        parsed["source"] = f"groq:{GROQ_MODEL}"
        parsed["status"] = "ok"
        GROQ_CIRCUIT["failures"] = 0
        return parsed

    except Exception as e:
        msg = str(e).lower()
        print(f"Groq failed: {e}")
        GROQ_CIRCUIT["failures"] += 1
        if GROQ_CIRCUIT["failures"] >= GROQ_THRESHOLD:
            GROQ_CIRCUIT["open_until"] = time.time() + GROQ_COOLDOWN
            print(f"⚠️  Groq circuit OPEN for {GROQ_COOLDOWN}s")
        return {"status": "fallback", "source": "groq_unavailable"}


print("Groq classifier ready ✓")

Groq client initialized ✓ (key len=56)
Groq classifier ready ✓


In [7]:
# ---------- Lawyer matcher with tiered relevance ----------
_lawyer_embeddings = None
LAWYER_CACHE_FILE = '../datasets/lawyer_embeddings.pkl'
DEFAULT_RERANK_POOL = 500

TIER_PRIMARY          = "primary"
TIER_SECONDARY_STRONG = "secondary_strong"
TIER_SECONDARY_WEAK   = "secondary_weak"

MIN_SCORE_SECONDARY_WEAK = 0.45


def _build_lawyer_embeddings():
    global _lawyer_embeddings
    if _lawyer_embeddings is not None:
        return

    if os.path.exists(LAWYER_CACHE_FILE):
        with open(LAWYER_CACHE_FILE, "rb") as f:
            cached = pickle.load(f)
        if cached.get("model") == EMBEDDING_MODEL_NAME and cached.get("n") == len(lawyers):
            _lawyer_embeddings = cached["embeddings"]
            print(f"Loaded cached lawyer embeddings ({_lawyer_embeddings.shape}) ✓")
            return

    texts = []
    for _, lw in lawyers.iterrows():
        parts = [
            str(lw.get("practice_area_primary", "")),
            str(lw.get("practice_area_secondary", "")),
            str(lw.get("city", "")),
            str(lw.get("experience_level", "")),
        ]
        texts.append(" | ".join(parts))

    print("Building lawyer embeddings (one-time) ...")
    _lawyer_embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    with open(LAWYER_CACHE_FILE, "wb") as f:
        pickle.dump({
            "model": EMBEDDING_MODEL_NAME,
            "n": len(lawyers),
            "embeddings": _lawyer_embeddings
        }, f)
    print(f"Saved lawyer embeddings → {LAWYER_CACHE_FILE} ✓")


def _classify_match(lawyer_row, domain: str):
    """
    Returns (score, tier) or (0.0, None) if the lawyer is not relevant.
    Tiers:
      - primary          : lawyer's primary is a primary area of the domain
      - secondary_strong : lawyer's primary is a secondary area of the domain
      - secondary_weak   : lawyer's secondary overlaps the domain's PRIMARY areas
    """
    tax = DOMAIN_TAXONOMY.get(domain, {"primary": [domain], "secondary": []})
    primary_areas = set(tax["primary"])
    secondary_areas = set(tax["secondary"])

    primary = str(lawyer_row.get('practice_area_primary', '')).strip()
    secondary_list = [
        s.strip()
        for s in str(lawyer_row.get('practice_area_secondary', '')).split(';')
        if s.strip()
    ]

    rating = float(lawyer_row.get('profile_rating', 4.0) or 4.0)
    rating_bonus = (rating / 5.0) * 0.2

    # Tier 1 — direct primary match
    if primary in primary_areas:
        return 1.0 + rating_bonus, TIER_PRIMARY

    # Tier 2 — lawyer's primary is a secondary area of this domain
    if primary in secondary_areas:
        return 0.7 + rating_bonus, TIER_SECONDARY_STRONG

    # Tier 3 — lawyer's secondary overlaps the domain's PRIMARY areas only
    sec_primary_matches = sum(1 for s in secondary_list if s in primary_areas)
    if sec_primary_matches > 0:
        score = min(0.35, sec_primary_matches * 0.15) + rating_bonus   # ← the missing line
        if score >= MIN_SCORE_SECONDARY_WEAK:
            return score, TIER_SECONDARY_WEAK

    # No relevant match
    return 0.0, None


def match_lawyers_by_domain(
    domain: str,
    user_prompt: str = "",
    top_k: int = None,
    rerank_pool: int = DEFAULT_RERANK_POOL,
) -> list:
    _build_lawyer_embeddings()

    tax = DOMAIN_TAXONOMY.get(domain, {"primary": [domain], "secondary": []})
    primary_areas = set(tax["primary"])
    secondary_areas = set(tax["secondary"])
    all_relevant = primary_areas | secondary_areas

    lp_primary = lawyers['practice_area_primary'].fillna('').astype(str).str.strip()
    lp_secondary = lawyers['practice_area_secondary'].fillna('').astype(str)

    mask_primary = lp_primary.isin(primary_areas)
    mask_primary_sec = lp_primary.isin(secondary_areas) & ~mask_primary

    def _sec_overlap(s: str) -> bool:
        parts = {p.strip() for p in s.split(';') if p.strip()}
        return bool(parts & all_relevant)

    mask_secondary = lp_secondary.apply(_sec_overlap) & ~mask_primary & ~mask_primary_sec

    matched_mask = mask_primary | mask_primary_sec | mask_secondary
    matched_df = lawyers[matched_mask].copy()

    if matched_df.empty:
        return []

    rows = []
    for idx, row in matched_df.iterrows():
        sc, tier = _classify_match(row, domain)
        if sc > 0 and tier is not None:
            rows.append((sc, int(idx), row, tier))

    rows.sort(key=lambda x: x[0], reverse=True)
    pool = rows if rerank_pool is None else rows[:rerank_pool]

    if user_prompt and pool:
        q_vec = _cached_query_embedding(user_prompt)
        pool_idx = [p[1] for p in pool]
        pool_embs = _lawyer_embeddings[pool_idx]
        sims = util.cos_sim(q_vec, pool_embs).numpy()[0]
        final = [
            (t + float(s) * 0.5, lw, tier, idx_global)
            for (t, idx_global, lw, tier), s in zip(pool, sims)
        ]
        if rerank_pool is not None and len(rows) > rerank_pool:
            tail = [(t, lw, tier, idx_global) for (t, idx_global, lw, tier) in rows[rerank_pool:]]
            final.extend(tail)
    else:
        final = [(t, lw, tier, idx_global) for (t, idx_global, lw, tier) in rows]

    TIER_ORDER = {TIER_PRIMARY: 0, TIER_SECONDARY_STRONG: 1, TIER_SECONDARY_WEAK: 2}
    final.sort(key=lambda x: (TIER_ORDER[x[2]], -x[0]))

    selected = final if top_k is None else final[:top_k]

    results = []
    for sc, lawyer, tier, _ in selected:
        results.append({
            "advocate_name": lawyer.get('advocate_name'),
            "city": lawyer.get('city'),
            "practice_area_primary": lawyer.get('practice_area_primary'),
            "practice_area_secondary": lawyer.get('practice_area_secondary'),
            "experience_level": lawyer.get('experience_level'),
            "profile_rating": lawyer.get('profile_rating'),
            "consultation_fee_inr": lawyer.get('consultation_fee_inr'),
            "match_type": tier,
            "score": round(sc, 3),
        })
    return results


def match_lawyers_local(user_prompt: str, top_k: int = None) -> dict:
    intent = vector_extract_intent(user_prompt)
    domain = intent["primary_domain"]
    matched = match_lawyers_by_domain(domain, user_prompt=user_prompt, top_k=top_k)
    return {
        "query": user_prompt,
        "detected_domain": domain,
        "confidence": intent.get("confidence", 0.0),
        "margin": intent.get("margin", 0.0),
        "ambiguous": intent.get("ambiguous", False),
        "top3_domains": intent.get("top3_domains", []),
        "language": intent.get("language", "english"),
        "latency_ms": intent.get("latency_ms", 0.0),
        "intent_source": intent.get("source", "local_vector_embedding"),
        "recommended_lawyers": matched,
        "total_lawyers": len(matched),
    }


def match_lawyers_with_llm(user_prompt: str, llm_result: dict, top_k: int = None) -> dict:
    """Generic — works for both Gemini and Groq results."""
    domain = llm_result["primary_domain"]
    matched = match_lawyers_by_domain(domain, user_prompt=user_prompt, top_k=top_k)
    return {
        "query": user_prompt,
        "detected_domain": domain,
        "confidence": llm_result.get("confidence", 0.0),
        "margin": None,
        "ambiguous": False,
        "top3_domains": [(domain, llm_result.get("confidence", 0.0))],
        "language": llm_result.get("language", "unknown"),
        "latency_ms": None,
        "intent_source": llm_result.get("source", "llm"),
        "recommended_lawyers": matched,
        "total_lawyers": len(matched),
    }


print("Lawyer matcher with tiered relevance ready ✓")

Lawyer matcher with tiered relevance ready ✓


In [8]:
# ---------- Router: Local → Gemini → Groq → Local-with-clarification ----------
LANGUAGE_THRESHOLDS = {
    "english":    {"confidence": 0.55, "margin": 0.05},
    "hinglish":   {"confidence": 0.50, "margin": 0.05},
    "devanagari": {"confidence": 0.42, "margin": 0.03},
}


def final_route(user_prompt: str, top_k: int = None) -> dict:
    # ---- Step 1: local ----
    local = match_lawyers_local(user_prompt, top_k=top_k)

    lang = local.get("language", "english")
    th = LANGUAGE_THRESHOLDS.get(lang, LANGUAGE_THRESHOLDS["english"])
    is_special = local.get("intent_source") == "special_route"

    confident = is_special or (
        local["confidence"] >= th["confidence"]
        and not local["ambiguous"]
        and local.get("margin", 0) >= th["margin"]
    )

    if confident:
        local["route_used"] = "special_route" if is_special else "local_confident"
        local["threshold_used"] = th
        return local

    # ---- Step 2: Gemini ----
    if check_gemini_available():
        print(f"Local unsure (lang={lang}, conf={local['confidence']}) → Gemini ...")
        gem = extract_intent_with_gemini(user_prompt)

        if gem.get("status") == "ok":
            result = match_lawyers_with_llm(user_prompt, gem, top_k=top_k)
            result["route_used"] = "gemini_fallback"
            result["threshold_used"] = th
            return result

        print("Gemini failed → trying Groq ...")

    # ---- Step 3: Groq ----
    if check_groq_available():
        grq = extract_intent_with_groq(user_prompt)

        if grq.get("status") == "ok":
            result = match_lawyers_with_llm(user_prompt, grq, top_k=top_k)
            result["route_used"] = "groq_fallback"
            result["threshold_used"] = th
            return result

        print("Groq failed too → falling back to local")

    # ---- Step 4: local with clarification ----
    local["needs_clarification"] = True
    local["route_used"] = "local_unsure"
    local["threshold_used"] = th
    return local


print("Final router ready ✓ (Local → Gemini → Groq → Local)")

Final router ready ✓ (Local → Gemini → Groq → Local)


In [9]:
# ---------- QUICK SINGLE PROMPT TEST ----------
test_prompt = "mere pati ne mujhe maara aur ghar se nikaal diya"  # 👈 edit

# Set top_k=None to see ALL; set top_k=10 for a shorter list
result = final_route(test_prompt, top_k=10) #this limit to 10 lawyers is optional, you can remove it to see all matched lawyers

print(f"Query      : {test_prompt}")
print(f"Domain     : {result['detected_domain']}")
print(f"Source     : {result['intent_source']}")
print(f"Language   : {result.get('language')}")
print(f"Confidence : {result['confidence']}  |  Margin: {result.get('margin')}")
print(f"Route      : {result.get('route_used')}")
if result.get("needs_clarification"):
    print("⚠️  Needs user clarification")

lawyers_list = result["recommended_lawyers"]
primary_n = sum(1 for lw in lawyers_list if lw.get("match_type") == "primary")
secondary_n = sum(1 for lw in lawyers_list if lw.get("match_type") == "secondary")
print(f"\nTotal lawyers: {len(lawyers_list)} (primary: {primary_n}, secondary: {secondary_n})")
print("\nAll matched lawyers:")
for i, lw in enumerate(lawyers_list, 1):
    print(f"  {i:>3}. [{lw['match_type']:<9}] {lw['advocate_name']} ({lw['city']}) | "
          f"{lw['practice_area_primary']} | {lw['experience_level']} | "
          f"₹{lw['consultation_fee_inr']} | score {lw['score']}")

Loaded cached lawyer embeddings ((7000, 384)) ✓
Query      : mere pati ne mujhe maara aur ghar se nikaal diya
Domain     : Criminal Law
Source     : special_route
Language   : hinglish
Confidence : 0.99  |  Margin: 1.0
Route      : special_route

Total lawyers: 10 (primary: 10, secondary: 0)

All matched lawyers:
    1. [primary  ] Adv. Naveen Srivastava (Guwahati) | Criminal Law | 16-25 years | ₹750 | score 1.317
    2. [primary  ] Adv. Sakshi Arora (Ahmedabad) | Criminal Law | 16-25 years | ₹2000 | score 1.316
    3. [primary  ] Adv. Akash Bhatia (Kochi) | Criminal Law | 16-25 years | ₹500 | score 1.314
    4. [primary  ] Adv. Pranav Sharma (Lucknow) | Criminal Law | 26+ years | ₹2500 | score 1.314
    5. [primary  ] Adv. Ishaan Das (Ahmedabad) | Criminal Law | 8-15 years | ₹500 | score 1.314
    6. [primary  ] Adv. Aditya Bose (Chennai) | Criminal Law | 16-25 years | ₹1200 | score 1.313
    7. [primary  ] Adv. Anand Jain (Mumbai) | Criminal Law | 16-25 years | ₹1000 | score 1.313
  

In [ ]:
demos = [
   # 1. Popular Hinglish — most common real-world query
    "mera landlord security deposit wapas nahi de raha",

    # 2. Popular Devanagari — tests transliteration layer
    "मेरे बैंक अकाउंट से पैसे किसी ने ऑनलाइन फ्रॉड करके निकाल लिए",

    # 3. Popular English — textbook labour dispute
    "my company fired me without notice and refused to pay gratuity",

    # 4. Popular English — high-volume MACT query
    "car hit my scooter on highway and broke my leg, need compensation",

    # 5. Rare domain — IP / startup
    "my startup co-founder is stealing our source code and selling it to a competitor",

    # 6. Edge case — overlaps Employment + Data Privacy + Cyber
    "my employer leaked my personal data to a third party",

    # 7. Edge case — overlap Motor Accident + Consumer + Insurance
    "insurance company is refusing to pay my accident claim",

    # 8. Complex multi-issue — combines three problems
    "my husband is abusing me, has taken all my stridhan, and is threatening divorce",

    # 9. Language mix — Hinglish + Devanagari + English
    "meri salary नहीं मिली, kya karu",

    # 10. Vague / short — tests minimum input handling
    "divorce",
]

for demo in demos:
    r = final_route(demo)   # no top_k → all lawyers
    print(f"\nQuery: {demo}")
    print(f"Domain: {r['detected_domain']}")
    print(f"Source: {r['intent_source']}")
    print(f"Lang: {r.get('language')} | Conf: {r['confidence']} | Margin: {r.get('margin')}")
    print(f"Route: {r.get('route_used')}")
    print(f"Total lawyers matched: {len(r['recommended_lawyers'])}")
    print("Top 5 preview:")
    for lw in r['recommended_lawyers'][:5]:
        print(f"  - {lw['advocate_name']} ({lw['city']}) | {lw['practice_area_primary']} | "
              f"{lw['experience_level']} | ₹{lw['consultation_fee_inr']} | score {lw['score']}")


Query: mera landlord security deposit wapas nahi de raha
Domain: Real Estate & Housing
Source: local_vector_embedding
Lang: hinglish | Conf: 0.9554 | Margin: 0.1965
Route: local_confident
Total lawyers matched: 501
Top 5 preview:
  - Adv. Akshay Ghosh (Panaji) | Real Estate Law | 26+ years | ₹1000 | score 1.455
  - Adv. Sneha Khanna (Patna) | Real Estate Law | 8-15 years | ₹2500 | score 1.448
  - Adv. Ishaan Ghosh (Kochi) | Real Estate Law | 16-25 years | ₹3000 | score 1.446
  - Adv. Abhishek Singh (Panaji) | Real Estate Law | 16-25 years | ₹500 | score 1.443
  - Adv. Aarav Ghosh (Raipur) | Real Estate Law | 8-15 years | ₹1000 | score 1.44

Query: मेरे बैंक अकाउंट से पैसे किसी ने ऑनलाइन फ्रॉड करके निकाल लिए
Domain: Cybercrime & IT
Source: local_vector_embedding
Lang: devanagari | Conf: 0.8104 | Margin: 0.0786
Route: local_confident
Total lawyers matched: 494
Top 5 preview:
  - Adv. Isha Bansal (Mysuru) | Cyber Law | 8-15 years | ₹1500 | score 1.348
  - Adv. Akanksha Sethi (Amritsar) |

In [ ]:
# ---------- QUICK SINGLE PROMPT TEST ----------
test_prompt = "pati ne daaru pikar mar dya"  # 👈 edit

# Set top_k=None to see ALL; set top_k=10 for a shorter list
result = final_route(test_prompt, top_k=None)

print(f"Query      : {test_prompt}")
print(f"Domain     : {result['detected_domain']}")
print(f"Source     : {result['intent_source']}")
print(f"Language   : {result.get('language')}")
print(f"Confidence : {result['confidence']}  |  Margin: {result.get('margin')}")
print(f"Route      : {result.get('route_used')}")
if result.get("needs_clarification"):
    print("⚠️  Needs user clarification")

lawyers_list = result["recommended_lawyers"]
primary_n = sum(1 for lw in lawyers_list if lw.get("match_type") == "primary")
secondary_n = sum(1 for lw in lawyers_list if lw.get("match_type") == "secondary")
print(f"\nTotal lawyers: {len(lawyers_list)} (primary: {primary_n}, secondary: {secondary_n})")
print("\nTop 10 matched lawyers:")
for i, lw in enumerate(lawyers_list[:10], 1):
    print(f"  {i:>3}. [{lw['match_type']:<9}] {lw['advocate_name']} ({lw['city']}) | "
          f"{lw['practice_area_primary']} | {lw['experience_level']} | "
          f"₹{lw['consultation_fee_inr']} | score {lw['score']}")
if len(lawyers_list) > 10:
    print(f"  ... and {len(lawyers_list) - 10} more")

Query      : mere pati ne mujhe maara aur ghar se nikaal diya
Domain     : Family & Succession
Source     : special_route
Language   : hinglish
Confidence : 0.99  |  Margin: 1.0
Route      : special_route

Total lawyers: 530 (primary: 253, secondary: 0)

Top 10 matched lawyers:
    1. [primary  ] Adv. Akash Desai (Lucknow) | Family Law | 16-25 years | ₹2000 | score 1.322
    2. [primary  ] Adv. Anil Kumar (Mumbai) | Family Law | 8-15 years | ₹2000 | score 1.32
    3. [primary  ] Adv. Charu Mehta (Guwahati) | Family Law | 4-7 years | ₹1500 | score 1.317
    4. [primary  ] Adv. Divya Dutta (Guwahati) | Family Law | 8-15 years | ₹2500 | score 1.316
    5. [primary  ] Adv. Vaishnavi Sharma (Guwahati) | Family Law | 8-15 years | ₹2500 | score 1.316
    6. [primary  ] Adv. Aadhya Ghosh (New Delhi) | Family Law | 4-7 years | ₹750 | score 1.316
    7. [primary  ] Adv. Ashish Joshi (Guwahati) | Family Law | 2-3 years | ₹5000 | score 1.314
    8. [primary  ] Adv. Aparna Naik (Raipur) | Family La

In [ ]:
# ---------- Accuracy evaluation on test_queries.csv ----------
def evaluate_local_classifier():
    if "legal_domain" not in queries.columns:
        print("⚠️  No 'legal_domain' column in test_queries.csv — skipping eval")
        return

    correct = 0
    total = 0
    ambiguous_count = 0
    by_lang = {"english": [0, 0], "hinglish": [0, 0], "devanagari": [0, 0]}
    latencies = []

    for _, row in queries.iterrows():
        prompt = str(row.get("query") or row.get("text") or "")
        gold = str(row.get("legal_domain", "")).strip()
        if not prompt or not gold:
            continue

        out = vector_extract_intent(prompt)
        total += 1
        latencies.append(out["latency_ms"])
        if out["ambiguous"]:
            ambiguous_count += 1

        lang = out.get("language", "english")
        by_lang.setdefault(lang, [0, 0])
        by_lang[lang][1] += 1
        if out["primary_domain"] == gold:
            correct += 1
            by_lang[lang][0] += 1

    if total == 0:
        print("No valid rows.")
        return

    print(f"\n=== Local Classifier Evaluation ===")
    print(f"Total queries           : {total}")
    print(f"Overall accuracy        : {correct/total:.2%}")
    print(f"Ambiguous (margin<0.05) : {ambiguous_count/total:.2%}")
    print(f"Avg latency (ms)        : {np.mean(latencies):.2f}")
    print(f"P95 latency (ms)        : {np.percentile(latencies, 95):.2f}")
    print("\nAccuracy by language:")
    for lang, (c, t) in by_lang.items():
        if t:
            print(f"  {lang:<12}: {c}/{t} = {c/t:.2%}")

evaluate_local_classifier()

No valid rows.


## Summary of Improvements

### Language handling
- Devanagari keywords added to BOOSTS per domain (60+ terms)
- Transliteration map (60+ Devanagari → Latin) applied before keyword boost
- Language detection: english / hinglish / devanagari
- Hinglish synonyms baked into EXPANDED_DESCRIPTIONS for every domain

### Router
- Local always runs first
- Language-aware confidence thresholds (English 0.55, Hinglish 0.50, Devanagari 0.42)
- Gemini only fires when local falls below threshold
- Circuit breaker prevents repeated Gemini calls when down

### Cache
- Versioned by model + taxonomy + boosts (auto-invalidates on any edit)
- Lawyer embeddings cached to disk (60s one-time build → 1s every run after)

### Testing
- Cell 9: batch demo
- Cell 10: quick single-prompt test
- Cell 11: accuracy evaluation on test_queries.csv, broken down by language

### To test new prompts
Run only **Cell 9** (batch) or **Cell 10** (single). No other cells need re-running.
